# **Integrantes**

**Ângelo Malta Reina - RM 570769**

**Gustavo Mendonça Duarte - RM 570561**

In [81]:
!pip install openai

# RAG

In [82]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph langchain-openai langchain-core pypdf unstructured docx2txt

In [83]:
import getpass
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore

In [84]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [85]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [86]:
vector_store = InMemoryVectorStore(embeddings)

In [87]:
import bs4
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader, Docx2txtLoader

file_path = "Dados da missão V3.docx"
loader10 = Docx2txtLoader(file_path)



docs10 = loader10.load()
docs = docs10

In [88]:
docs

[Document(metadata={'source': 'Dados da missão V3.docx'}, page_content='# matriz em python contendo os dados da missão:\n\ndados_da_missao = \n\n    [24, 92, 88, 96, 90],\n\n    [27, 80, 72, 94, 85],\n\n    [31, 65, 58, 91, 70],\n\n    [36, 42, 38, 87, 55],\n\n    [39, 28, 19, 78, 35],\n\n    [34, 55, 32, 82, 50]\n\n\n\nAs variações devem ocorrer relativamente com os dois valores mais próximos do valor obtido dentro do array no local designado.\n\n\n\nCada linha dentro da matriz é um ciclo.\n\nCada ciclo tem 5 valores.\n\nOs valores são considerados como colunas.\n\nOs valores são ordenados nesta ordem, respectivamente:\n\n[temperatura(°C), comunicação(%), bateria(%), oxigênio(%), estabilidade(%)]\n\nQuando falar os ciclos, não é necessário escrever a linha da matriz, apenas dite ‘Ciclo’, seguido do número correspondente e siga as informações.')]

In [89]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

print(f"Split pdf into {len(all_splits)} sub-documents.")

Split pdf into 1 sub-documents.


In [90]:
all_splits[0]



Document(metadata={'source': 'Dados da missão V3.docx', 'start_index': 0}, page_content='# matriz em python contendo os dados da missão:\n\ndados_da_missao = \n\n    [24, 92, 88, 96, 90],\n\n    [27, 80, 72, 94, 85],\n\n    [31, 65, 58, 91, 70],\n\n    [36, 42, 38, 87, 55],\n\n    [39, 28, 19, 78, 35],\n\n    [34, 55, 32, 82, 50]\n\n\n\nAs variações devem ocorrer relativamente com os dois valores mais próximos do valor obtido dentro do array no local designado.\n\n\n\nCada linha dentro da matriz é um ciclo.\n\nCada ciclo tem 5 valores.\n\nOs valores são considerados como colunas.\n\nOs valores são ordenados nesta ordem, respectivamente:\n\n[temperatura(°C), comunicação(%), bateria(%), oxigênio(%), estabilidade(%)]\n\nQuando falar os ciclos, não é necessário escrever a linha da matriz, apenas dite ‘Ciclo’, seguido do número correspondente e siga as informações.')

In [91]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['5626b095-e518-434b-ac3d-f8d3c085828c']


In [92]:
from langsmith import Client
client = Client(api_key=userdata.get('LANGSMITH_API_KEY'))
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True, dangerously_pull_public_prompt=True)

In [93]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [94]:
prompt.messages[0].prompt.template = '''You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say 'NAO SEI'. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:'''

In [95]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}


from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

## **CODIGO DO CHATBOT**

In [ ]:
from openai import OpenAI
from google.colab import userdata
import os


client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))


from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


embeddings = OpenAIEmbeddings(
    api_key=userdata.get('OPENAI_API_KEY')
)



vector_store = InMemoryVectorStore(embeddings)


file_path = "Dados da missão V3.docx"

loader = Docx2txtLoader(file_path)

docs = loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

all_splits = text_splitter.split_documents(docs)


vector_store.add_documents(all_splits)


SYSTEM_PROMPT = """

Você é um sistema integrado em um espaçonave chamado Mission Control AI. Sua tarefa é analisar com base no contexto fornecido, o status das condições operacionais da espaçonave e gerar um relatório do ciclo feito com base nas seguintes informações:


Analisa a temperatura interna do módulo (em °C).

    Regras:
      - Menor que 18 °C       → ATENÇÃO  (1 ponto) - frio demais
      - De 18 °C até 30 °C   → NORMAL   (0 pontos) - faixa ideal
      - De 31 °C até 35 °C   → ATENÇÃO  (1 ponto) - aquecendo
      - Maior que 35 °C      → CRÍTICO  (2 pontos) - superaquecimento


    Analisa a qualidade do sinal de comunicação (em %).

    Regras:
      - Menor que 30%   → CRÍTICO  (2 pontos) - sinal quase perdido
      - De 30% a 59%   → ATENÇÃO  (1 ponto) - sinal instável
      - 60% ou mais    → NORMAL   (0 pontos) - sinal adequado

    Analisa o nível de bateria da missão (em %).

    Regras:
      - Menor que 20%   → CRÍTICO  (2 pontos) - bateria crítica
      - De 20% a 49%   → ATENÇÃO  (1 ponto) - bateria baixa
      - 50% ou mais    → NORMAL   (0 pontos) - energia suficiente

    Analisa o nível de oxigênio disponível (em %).

    Regras:
      - Menor que 80%   → CRÍTICO  (2 pontos) - risco à vida
      - De 80% a 89%   → ATENÇÃO  (1 ponto) - oxigênio reduzido
      - 90% ou mais    → NORMAL   (0 pontos) - oxigênio adequado

    Analisa a estabilidade geral dos sistemas (em %).

    Regras:
      - Menor que 40%   → CRÍTICO  (2 pontos) - sistemas instáveis
      - De 40% a 69%   → ATENÇÃO  (1 ponto) - estabilidade reduzida
      - 70% ou mais    → NORMAL   (0 pontos) - sistemas estáveis

=====================================================================

Deve mostrar ações que devem ser tomadas caso esteja em ATENÇÃO ou CRÍTICO para cada area

Em caso de ATENÇÃO:

Temperatura: "Verificar controle termico"
Comunicação: "Verificar motivos de má comunicação"
Energia: "Revisar sistemas que estejam consumindo energia"
Oxigênio: "Assegurar funcionamento do protocolo de suporte á vida"
Estabilidade: "Analisar motivos da média estabilidade"

Em caso de CRÍTICO:

Temperatura: "Verificar controle termico da missão com urgência"
Comunicação: "Tentar restabelecer contato com a base mais cedo possível!"
Energia: "Ativer modo economia de energia!"
Oxigênio: "Acionar protocolo de suporte à vida"
Estabilidade: "Reduzir operações não essenciais"


Classifica o ciclo com base na pontuação total de risco.

Pontuação máxima possível por ciclo: 10 pontos (5 áreas × 2 pontos)

    Regras:
       0 a 2 pontos  → SITUAÇÃO ESTÁVEL
       3 a 5 pontos  → SITUAÇÃO INSTÁVEL
       6 a 10 pontos → SITUAÇÃO CRÍTICA

Quando falar os ciclos, não é necessário escrever a linha da matriz, apenas dite ‘Ciclo’, seguido do número correspondente.

Faça um resumo das ações necessarias ao final

"""


msgs = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]



while True:

    pergunta = input(" ")

    if pergunta.lower() in ["sair", "exit", "quit", "tchau", "break", "obrigado"]:
        break

    retrieved_docs = vector_store.similarity_search(
        pergunta,
        k=3
    )

    contexto = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )


    pergunta_com_contexto = f"""
    Pergunta do usuário:
    {pergunta}

    Contexto recuperado:
    {contexto}
    """

    msgs.append({
        "role": "user",
        "content": pergunta_com_contexto
    })


    resposta = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=msgs,
        temperature=0.3
    )

    resposta_texto = resposta.choices[0].message.content

    print(f"{resposta_texto}")

    msgs.append({
        "role": "assistant",
        "content": resposta_texto
    })

 Relatorio
**Relatório de Ciclos da Missão**

**Ciclo 1:**
- Temperatura: 24 °C → NORMAL (0 pontos)
- Comunicação: 92% → NORMAL (0 pontos)
- Bateria: 88% → NORMAL (0 pontos)
- Oxigênio: 96% → NORMAL (0 pontos)
- Estabilidade: 90% → NORMAL (0 pontos)

**Pontuação Total: 0 pontos → SITUAÇÃO ESTÁVEL**

---

**Ciclo 2:**
- Temperatura: 27 °C → NORMAL (0 pontos)
- Comunicação: 80% → ATENÇÃO (1 ponto) 
  - Ação: "Verificar motivos de má comunicação"
- Bateria: 72% → ATENÇÃO (1 ponto) 
  - Ação: "Revisar sistemas que estejam consumindo energia"
- Oxigênio: 94% → NORMAL (0 pontos)
- Estabilidade: 85% → NORMAL (0 pontos)

**Pontuação Total: 2 pontos → SITUAÇÃO ESTÁVEL**

---

**Ciclo 3:**
- Temperatura: 31 °C → ATENÇÃO (1 ponto) 
  - Ação: "Verificar controle termico"
- Comunicação: 65% → ATENÇÃO (1 ponto) 
  - Ação: "Verificar motivos de má comunicação"
- Bateria: 58% → ATENÇÃO (1 ponto) 
  - Ação: "Revisar sistemas que estejam consumindo energia"
- Oxigênio: 91% → NORMAL (0 pontos)
- Estabili